# Symmetric and asymmetric frustration analysis (V3)

This notebook produces the requested plots from the V3 asymmetric and symmetric W-bank files.

1. For a selected `γ`, `W` (default `W2`), and simulation: final tissue state by gene, intracellular frustration per cell, and intercellular frustration per cell.
2. At that selected `γ` and `W`: per-simulation distributions across cells, per-simulation mean frustration of the final tissue, and cell-count histograms by frustration density.
3. Across `γ` and the selected W signs/indices: final energy, unique expression profiles, mean intracellular frustration, and mean intercellular frustration.

The symmetric and asymmetric data use separate summary caches. Output plots are saved to `plot_dir`. Run cells from top to bottom after the remote symmetric job has finished writing its files.


In [ ]:
using Plots
using Random
using Statistics
using JLD2
using CodecZlib   # older JLD2 versions need this to read compressed files

# Lattice, Model and tissue_frustration (used to recompute per-sim final frustration)
include("/home/twwoodw/ondemand/script/ising_core_V3.jl")
include("/home/twwoodw/my_folder/compute/data/gamma_study/function_Gammarange_analysis_V3.jl")

println("Julia threads: ", Threads.nthreads())


In [ ]:
### ----------- data location ----------- ###
base_path      = "/data/scratch/projects/punim0638/woodward/nonrcep_casestudy/gamma_data_V3"
cache_file_asym = "/home/twwoodw/my_folder/compute/data/gamma_study/summary_gammarange_set1_V3_compare_asym.jld2"
cache_file_sym  = "/home/twwoodw/my_folder/compute/data/gamma_study/summary_gammarange_set1_V3_compare_sym.jld2"
cache_file      = cache_file_asym

function make_filename(base_path, W_name, β_name, σ_name, n_sims, max_sweeps)
    return joinpath(base_path,
        "$(W_name)_$(β_name)_$(σ_name)_$(n_sims)_max$(max_sweeps)sweeps_flipequ.jld2")
end

function make_filename_sym(base_path, W_name, β_name, σ_name, n_sims, max_sweeps)
    return joinpath(base_path,
        "$(W_name)_sym_$(β_name)_$(σ_name)_$(n_sims)_max$(max_sweeps)sweeps_flipequ.jld2")
end

### ----------- constants ----------- ###
n_cells    = 400
n_genes    = 9
n_sims     = 10
max_sweeps = 10_000

β_name = "betaInf"
σ_name = "σrand"

gammas      = [1.0, 0.8, 0.5, 0.3, 0.25, 0.22, 0.20, 0.18, 0.16, 0.14, 0.125, 0.12, 0.1, 0.08, 0.07, 0.06, 0.05, 0.04, 0.001, 0.0]
gamma_names = ["g1", "g08", "g05", "g03", "g025", "g022", "g02", "g018", "g016", "g014", "g0125", "g012", "g01", "g008", "g007", "g006", "g005", "g004", "g0001", "g0"]

asymmetric_grns = ["tree", "negtri", "ICFF", "CFF", "negtritwin", "negtritri"]
symmetric_grns = ["treesym", "negtrisym", "ICFFsym", "CFFsym", "negtritwinsym", "negtritrisym"]
base_grns = ["tree", "negtri", "ICFF", "CFF", "negtritwin", "negtritri"]
grns = asymmetric_grns

### ----------- which Ws ----------- ###
W_range = 1:1000
W_signs = [1]        # [1] = W only, [-1] = -W only, [1, -1] = both

W_name(i, sgn) = sgn == 1 ? "W$(i)" : "-W$(i)"

# Fixed-W figures below use these selections.
gamma_fixed = 0.3
W_fixed = 2
W_sign_fixed = 1
sim_fixed = 1

grn_colors = Dict(
    "tree"       => :orange,
    "negtri"     => :teal,
    "ICFF"       => :purple,
    "CFF"        => :violet,
    "negtritwin" => :blue,
    "negtritri"  => :cyan,
)


In [ ]:
all_files = [make_filename(base_path, W_name(i, s), β_name, σ_name, n_sims, max_sweeps)
             for i in W_range for s in W_signs]
files = filter(isfile, all_files)
println("$(length(files)) of $(length(all_files)) asymmetric files exist")

all_files_sym = [make_filename_sym(base_path, W_name(i, s), β_name, σ_name, n_sims, max_sweeps)
                 for i in W_range for s in W_signs]
files_sym = filter(isfile, all_files_sym)
println("$(length(files_sym)) of $(length(all_files_sym)) symmetric files exist")

isempty(files) && error("No asymmetric input files found in $base_path")
isempty(files_sym) && error("No symmetric input files found in $base_path")

@time S_asym = summarize_W_bank(files;
    gamma_names = gamma_names,
    grns        = asymmetric_grns,
    n_cells     = n_cells,
    n_sims      = n_sims,
    max_sweeps  = max_sweeps,
    cache_file  = cache_file_asym
)

@time S_sym = summarize_W_bank(files_sym;
    gamma_names = gamma_names,
    grns        = symmetric_grns,
    n_cells     = n_cells,
    n_sims      = n_sims,
    max_sweeps  = max_sweeps,
    cache_file  = cache_file_sym
)

S = S_asym;


In [ ]:
d_asym = load(cache_file_asym)
d_sym  = load(cache_file_sym)
println("asymmetric GRNs in cache: ", d_asym["settings"]["grns"])
println("symmetric GRNs in cache: ", d_sym["settings"]["grns"])

load_summary(path) = let d = load(path)
    (; (Symbol(k) => v for (k, v) in d)...)
end
S_asym = load_summary(cache_file_asym)
S_sym  = load_summary(cache_file_sym)
S = S_asym

for (summary, expected_grns, label) in ((S_asym, asymmetric_grns, "asymmetric"), (S_sym, symmetric_grns, "symmetric"))
    @assert summary.settings["gamma_names"] == collect(gamma_names) "$label gamma_names mismatch"
    @assert summary.settings["grns"] == collect(expected_grns) "$label GRN names mismatch"
    @assert summary.settings["n_sims"] == n_sims "$label n_sims mismatch"
    @assert summary.settings["n_cells"] == n_cells "$label n_cells mismatch"
    @assert summary.settings["max_sweeps"] == max_sweeps "$label max_sweeps mismatch"
end

println("Loaded asymmetric summary: $(length(S_asym.W_files)) Ws | version: $(S_asym.settings["summary_version"])")
println("Loaded symmetric summary: $(length(S_sym.W_files)) Ws | version: $(S_sym.settings["summary_version"])")


## Across-γ and across-W plots

Curves compare corresponding GRNs. Color denotes the GRN; solid/circle is asymmetric and dashed/diamond is symmetric. Error ribbons show the standard deviation across W after averaging simulations within each W. The γ axis is sorted numerically for readable curves, regardless of the run-list order.


In [ ]:
comparison_cases = [
    (label = "Asymmetric", summary = S_asym, names = asymmetric_grns, style = :solid, marker = :circle),
    (label = "Symmetric", summary = S_sym, names = symmetric_grns, style = :dash, marker = :diamond),
]

gamma_index(summary, gamma) = begin
    names = String.(summary.settings["gamma_names"])
    idx = findfirst(==(gamma_names[findfirst(==(gamma), gammas)]), names)
    idx === nothing && error("γ=$gamma is absent from summary gamma names $names")
    idx
end

grn_index(summary, name) = begin
    names = String.(summary.settings["grns"])
    idx = findfirst(==(name), names)
    idx === nothing && error("GRN $name is absent from summary GRNs $names")
    idx
end

function per_W_mean(summary, field, g_idx, k_idx)
    data = getfield(summary, field)
    ndims(data) == 4 || error("Expected a 4D summary field $field, got $(ndims(data))D")
    return vec(mean(data[:, :, g_idx, k_idx]; dims = 1))
end

function plot_across_gamma(field, ylabel, title, filename; skip_zero = false)
    gamma_plot = sort(skip_zero ? filter(!=(0.0), gammas) : copy(gammas))
    p = plot(xlabel = "γ", ylabel = ylabel, title = title,
             legend = :outerright, xrotation = 45, size = (1200, 650), dpi = 500)
    for (grn_pos, base_name) in enumerate(base_grns), case in comparison_cases
        k_idx = grn_index(case.summary, case.names[grn_pos])
        means = Float64[]
        stds = Float64[]
        for gamma in gamma_plot
            g_idx = gamma_index(case.summary, gamma)
            values_per_W = per_W_mean(case.summary, field, g_idx, k_idx)
            push!(means, mean(values_per_W))
            push!(stds, length(values_per_W) > 1 ? std(values_per_W) : 0.0)
        end
        plot!(p, gamma_plot, means; ribbon = stds, fillalpha = 0.10,
              color = grn_colors[base_name], linestyle = case.style, marker = case.marker,
              linewidth = 2, label = "$(base_name) — $(case.label)")
    end
    display(p)
    savefig(p, joinpath(plot_dir, filename))
    return p
end

plot_across_gamma(:final_E, "Final tissue energy", "Final energy vs γ", "final_energy_vs_gamma.png"; skip_zero = true)
plot_across_gamma(:n_profiles, "Unique expression profiles in final tissue", "Unique expression profiles vs γ", "unique_expression_profiles_vs_gamma.png"; skip_zero = true)
plot_across_gamma(:local_frus, "Mean intracellular frustration density", "Mean intracellular frustration density vs γ", "intracellular_frustration_vs_gamma.png"; skip_zero = true)
plot_across_gamma(:final_frus, "Mean intercellular frustration density", "Mean intercellular frustration density vs γ", "intercellular_frustration_vs_gamma.png"; skip_zero = true)


## Fixed γ and W: final states and cell-wise frustration maps

The selection values are set in the settings cell. The code below reads the selected asymmetric and `_sym_` W2 files directly and uses the existing per-cell frustration helpers.


In [ ]:
function cell_frustration_one_W(file, gamma_name; grns, n_cells)
    lat = Lattice(n_cells; n_neighbors = 8)
    out = Dict{String,NamedTuple}()

    jldopen(file, "r") do f
        W = f["meta/W"]
        gamma = Dict(zip(f["meta/gamma_names"], f["meta/gamma_values"]))[gamma_name]
        Kmap = f["meta/K"]
        sW = sign.(gamma .* W)
        nsum = zeros(Int, size(W, 1), n_cells)
        lu = zeros(Int, n_cells)
        gu = zeros(Int, n_cells)
        gn = zeros(Int, n_cells)

        for grn in grns
            sigma_finals = f["$(gamma_name)/$(grn)"]["sigma_finals_selfish"]
            Kp = grn_pairs(Kmap[grn])
            nE = length(Kp)
            local_entries = Vector{Vector{Float64}}()
            global_entries = Vector{Vector{Float64}}()

            for sim in eachindex(sigma_finals)
                local_cell_unsat!(lu, sigma_finals[sim], Kp)
                global_cell_unsat!(gu, gn, sigma_finals[sim], sW, lat, nsum)
                push!(local_entries, nE == 0 ? zeros(n_cells) : lu ./ nE)
                push!(global_entries, [gn[c] == 0 ? 0.0 : gu[c] / gn[c] for c in 1:n_cells])
            end

            out[grn] = (
                sigma_finals = sigma_finals,
                local_sim_entries = local_entries,
                global_sim_entries = global_entries,
                local_means = mean.(local_entries),
                global_means = mean.(global_entries),
            )
        end
    end
    return out
end

# Profile IDs depend only on the gene-state vector, so this key is shared by every panel.
configuration_id(profile) = foldl((id, gene_state) -> 2id + (gene_state == 1 ? 1 : 0), profile; init = 0) + 1
profile_color_key = palette(:tab20, 2^n_genes)

function final_tissue_heatmap(W_test, gamma_test, sim_test; grn, symmetric = false)
    gamma_idx = findfirst(==(gamma_test), gammas)
    gamma_idx === nothing && error("gamma_test=$gamma_test is not in gammas")
    gamma_name = gamma_names[gamma_idx]

    # Accept the short display names and the GRN names stored in the result files.
    saved_grn = if symmetric
        get(Dict("treesym" => "treesym", "trisym" => "negtrisym", "negtrisym" => "negtrisym",
                 "ICFFsym" => "ICFFsym", "CFFsym" => "CFFsym",
                 "negtwintrisym" => "negtritwinsym", "negtritwinsym" => "negtritwinsym",
                 "negtritrisym" => "negtritrisym"), grn) do
            error("Unknown symmetric GRN: $grn")
        end
    else
        get(Dict("tree" => "tree", "tri" => "negtri", "negtri" => "negtri",
                 "ICFF" => "ICFF", "CFF" => "CFF",
                 "negtwintri" => "negtritwin", "negtritwin" => "negtritwin",
                 "negtritri" => "negtritri"), grn) do
            error("Unknown asymmetric GRN: $grn")
        end
    end

    W_label = W_name(W_test, W_sign_fixed)
    file = symmetric ?
        make_filename_sym(base_path, W_label, β_name, σ_name, n_sims, max_sweeps) :
        make_filename(base_path, W_label, β_name, σ_name, n_sims, max_sweeps)
    isfile(file) || error("Result file not found: $file")

    sigma = jldopen(file, "r") do f
        group_path = "$(gamma_name)/$(saved_grn)"
        haskey(f, group_path) || error("Group $group_path is not in $file")
        sigma_finals = f[group_path]["sigma_finals_selfish"]
        sim_test in eachindex(sigma_finals) || error("Simulation $sim_test is not in $group_path")
        sigma_finals[sim_test]
    end

    n_tissue_cells, n_tissue_genes = size(sigma)
    side = isqrt(n_tissue_cells)
    side^2 == n_tissue_cells || error("$n_tissue_cells cells cannot be arranged in a square lattice")
    n_tissue_genes == n_genes || error("Expected $n_genes genes, found $n_tissue_genes")
    profiles = [configuration_id(Tuple(sigma[cell, :])) for cell in 1:n_tissue_cells]
    tissue = reshape(profiles, side, side)

    return heatmap(tissue;
        c = profile_color_key,
        clims = (0.5, 2^n_genes + 0.5),
        aspect_ratio = :equal,
        colorbar = false,
        title = "$(symmetric ? \"Sym\" : \"Asym\") $(grn)")
end

W_test = W_fixed
gamma_test = gamma_fixed
sim_test = sim_fixed
W_sign_fixed = 1  # Use -1 to plot the -W file instead.

fixed_cases = [
    (label = "Asymmetric", grns = asymmetric_grns,
     file = make_filename(base_path, W_name(W_test, W_sign_fixed), β_name, σ_name, n_sims, max_sweeps)),
    (label = "Symmetric", grns = symmetric_grns,
     file = make_filename_sym(base_path, W_name(W_test, W_sign_fixed), β_name, σ_name, n_sims, max_sweeps)),
]
fixed_data = Dict{String,Dict{String,NamedTuple}}()
for case in fixed_cases
    isfile(case.file) || error("Missing $(case.label) file: $(case.file)")
    fixed_data[case.label] = cell_frustration_one_W(case.file, gamma_names[findfirst(==(gamma_test), gammas)];
        grns = case.grns, n_cells = n_cells)
end

p1 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "tree")
p2 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "tri")
p3 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "ICFF")
p4 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "CFF")
p5 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "negtwintri")
p6 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "negtritri")

p7 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "treesym", symmetric = true)
p8 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "trisym", symmetric = true)
p9 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "ICFFsym", symmetric = true)
p10 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "CFFsym", symmetric = true)
p11 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "negtwintrisym", symmetric = true)
p12 = final_tissue_heatmap(W_test, gamma_test, sim_test; grn = "negtritrisym", symmetric = true)

p_final = plot(p1, p2, p3, p4, p5, p6, p7, p8, p9, p10, p11, p12;
    layout = (2, 6),
    size = (1800, 650),
    plot_title = "Final tissue profiles: W$(W_test), γ=$(gamma_test), simulation $(sim_test)")
display(p_final)
savefig(p_final, joinpath(plot_dir,
    "final_tissue_profiles_W$(W_test)_$(gamma_names[findfirst(==(gamma_test), gammas)])_sim$(sim_test).png"))


## Fixed γ and W: distributions and per-simulation tissue means

Violin distributions show cell-level frustration for each simulation. The mean plots show the average across cells in each final tissue.


In [ ]:
function plot_cell_distributions(metric, ylabel, title, filename)
    panels = [begin
        p = plot(title = base_grn, xlabel = "Simulation", ylabel = ylabel,
                 xticks = 1:n_sims, xlims = (0.4, n_sims + 0.6), ylims = (0, 1),
                 legend = base_idx == 1 ? :best : false)
        for case in fixed_cases
            source_name = case.grns[base_idx]
            values_by_sim = getproperty(fixed_data[case.label][source_name], metric)
            offset = case.label == "Asymmetric" ? -0.16 : 0.16
            for sim in eachindex(values_by_sim)
                label = base_idx == 1 && sim == 1 ? case.label : ""
                violin!(p, fill(sim + offset, length(values_by_sim[sim])), values_by_sim[sim];
                    color = grn_colors[base_grn], alpha = case.label == "Asymmetric" ? 0.5 : 0.28,
                    linewidth = 0.6, label = label)
            end
        end
        p
    end for (base_idx, base_grn) in enumerate(base_grns)]
    p = plot(panels...; layout = (2, 3), size = (1300, 800), dpi = 500,
             plot_title = "$(title): $(selected_W_label), γ=$gamma_fixed, all simulations")
    display(p)
    savefig(p, joinpath(plot_dir, filename))
    return p
end

function plot_tissue_mean_by_sim(metric, ylabel, title, filename)
    panels = [begin
        p = plot(title = base_grn, xlabel = "Simulation", ylabel = ylabel,
                 xticks = 1:n_sims, xlims = (0.5, n_sims + 0.5), ylims = (0, 1),
                 legend = base_idx == 1 ? :best : false)
        for case in fixed_cases
            result = fixed_data[case.label][case.grns[base_idx]]
            values_by_sim = getproperty(result, metric)
            sim_means = mean.(values_by_sim)
            plot!(p, 1:length(sim_means), sim_means; color = grn_colors[base_grn],
                  linestyle = case.label == "Asymmetric" ? :solid : :dash, marker = case.label == "Asymmetric" ? :circle : :diamond, linewidth = 2,
                  label = base_idx == 1 ? case.label : "")
            println("$(case.label) $(base_grn): mean final-tissue frustration = $(round(mean(sim_means); digits=5))")
        end
        p
    end for (base_idx, base_grn) in enumerate(base_grns)]
    p = plot(panels...; layout = (2, 3), size = (1300, 800), dpi = 500,
             plot_title = "$(title): per-simulation final-tissue mean, $(selected_W_label), γ=$gamma_fixed")
    display(p)
    savefig(p, joinpath(plot_dir, filename))
    return p
end

plot_cell_distributions(:local_sim_entries, "Intracellular frustration density", "Intracellular frustration across cells", "intracellular_cell_distributions_$(selected_W_label)_$(selected_gamma_name).png")
plot_tissue_mean_by_sim(:local_sim_entries, "Mean intracellular frustration density", "Intracellular frustration", "intracellular_tissue_means_$(selected_W_label)_$(selected_gamma_name).png")
plot_cell_distributions(:global_sim_entries, "Intercellular frustration density", "Intercellular frustration across cells", "intercellular_cell_distributions_$(selected_W_label)_$(selected_gamma_name).png")
plot_tissue_mean_by_sim(:global_sim_entries, "Mean intercellular frustration density", "Intercellular frustration", "intercellular_tissue_means_$(selected_W_label)_$(selected_gamma_name).png")


## Fixed γ and W: histograms of cell counts by frustration density

These histograms count the number of cells in each frustration-density bin, pooling the simulations for the selected W and γ.


In [ ]:
function plot_cell_count_histograms(metric, xlabel, title, filename)
    bin_edges = 0.0:0.02:1.0
    panels = [begin
        p = plot(title = base_grn, xlabel = xlabel, ylabel = "Cell-simulation count per bin",
                 xlims = (0, 1), legend = base_idx == 1 ? :best : false)
        for case in fixed_cases
            source_name = case.grns[base_idx]
            values = reduce(vcat, getproperty(fixed_data[case.label][source_name], metric))
            histogram!(p, values; bins = bin_edges, color = grn_colors[base_grn],
                alpha = case.label == "Asymmetric" ? 0.48 : 0.26,
                linecolor = grn_colors[base_grn], label = base_idx == 1 ? case.label : "")
        end
        p
    end for (base_idx, base_grn) in enumerate(base_grns)]
    p = plot(panels...; layout = (2, 3), size = (1300, 800), dpi = 500,
             plot_title = "$(title): $(selected_W_label), γ=$gamma_fixed, all simulations")
    display(p)
    savefig(p, joinpath(plot_dir, filename))
    return p
end

plot_cell_count_histograms(:local_sim_entries, "Intracellular frustration density", "Cells per intracellular frustration bin", "intracellular_cell_count_histograms_$(selected_W_label)_$(selected_gamma_name).png")
plot_cell_count_histograms(:global_sim_entries, "Intercellular frustration density", "Cells per intercellular frustration bin", "intercellular_cell_count_histograms_$(selected_W_label)_$(selected_gamma_name).png")


In [ ]:
using LinearAlgebra
include("../Model/OG/functions/functions_frustration_index_OG.jl")

function cell_frustration_for_sim(W_idx, gamma, sim; grn, W_sign = 1, symmetric = false)
    gamma_idx = findfirst(==(gamma), gammas)
    gamma_idx === nothing && error("gamma=$gamma is not in gammas")
    gamma_name = gamma_names[gamma_idx]
    W_label = W_name(W_idx, W_sign)

    saved_grn = if symmetric
        get(Dict("treesym" => "treesym", "trisym" => "negtrisym", "negtrisym" => "negtrisym",
                 "ICFFsym" => "ICFFsym", "CFFsym" => "CFFsym",
                 "negtwintrisym" => "negtritwinsym", "negtritwinsym" => "negtritwinsym",
                 "negtritrisym" => "negtritrisym"), grn) do
            error("Unknown symmetric GRN: $grn")
        end
    else
        get(Dict("tree" => "tree", "tri" => "negtri", "negtri" => "negtri",
                 "ICFF" => "ICFF", "CFF" => "CFF",
                 "negtwintri" => "negtritwin", "negtritwin" => "negtritwin",
                 "negtritri" => "negtritri"), grn) do
            error("Unknown asymmetric GRN: $grn")
        end
    end

    file = symmetric ?
        make_filename_sym(base_path, W_label, β_name, σ_name, n_sims, max_sweeps) :
        make_filename(base_path, W_label, β_name, σ_name, n_sims, max_sweeps)
    isfile(file) || error("Result file not found: $file")

    sigma, W, K = jldopen(file, "r") do f
        group = "$(gamma_name)/$(saved_grn)"
        haskey(f, group) || error("Group $group is not in $file")
        sigma_finals = f[group]["sigma_finals_selfish"]
        sim in eachindex(sigma_finals) || error("Simulation $sim is not present in $group")
        (sigma_finals[sim], f["meta/W"], f["meta/K"][saved_grn])
    end

    # Intracellular density, using the existing single-cell function.
    intracellular = [
        intracellular_frustration_density(K, sigma[cell, :])[2]
        for cell in 1:n_cells
    ]

    # Build the same periodic 8-neighbor interaction matrix used by the model,
    # then use the existing state-dependent and per-cell intercellular functions.
    J_cell = zeros(Float64, n_cells, n_cells)
    for cell in 1:n_cells, neighbor in eight_neighbors_indices(cell, n_cells)
        J_cell[cell, neighbor] = 1.0
    end
    J_system = kron(Matrix{Float64}(I, n_cells, n_cells), K) +
               kron(J_cell, gamma .* W)
    J_frus = state_dept_frustration_idx_faster(
        Matrix{Float64}(J_system), Matrix{Float64}(sigma);
        n_genes = n_genes, n_cells = n_cells)
    intercellular = [
        intercellular_frustration_density(cell, J_frus;
            n_cells = n_cells, n_genes = n_genes)
        for cell in 1:n_cells
    ]

    return (; sigma, intracellular, intercellular)
end

function plot_cell_frustration_maps(W_idx, gamma, sim; grn, W_sign = 1, symmetric = false)
    result = cell_frustration_for_sim(W_idx, gamma, sim;
        grn = grn, W_sign = W_sign, symmetric = symmetric)

    side = isqrt(n_cells)
    side^2 == n_cells || error("n_cells=$n_cells cannot be arranged in a square lattice")
    W_label = W_name(W_idx, W_sign)
    case_label = symmetric ? "Symmetric" : "Asymmetric"

    p_intracellular = heatmap(reshape(result.intracellular, side, side);
        c = :viridis, clims = (0, 1), aspect_ratio = :equal,
        title = "Intracellular frustration", colorbar_title = "Density",
        grid = false, framestyle = :none)
    p_intercellular = heatmap(reshape(result.intercellular, side, side);
        c = :viridis, clims = (0, 1), aspect_ratio = :equal,
        title = "Intercellular frustration", colorbar_title = "Density",
        grid = false, framestyle = :none)

    p = plot(p_intracellular, p_intercellular;
        layout = (1, 2), size = (1000, 500),
        plot_title = "$case_label $grn — $W_label, γ=$gamma, sim=$sim")
    display(p)
    return p
end

# Example
frustration_by_cell = cell_frustration_for_sim(2, 0.3, 10; grn = "tree")
frustration_maps = plot_cell_frustration_maps(2, 0.3, 10; grn = "tree")
